# Herding and Liquidity in Order-Book Markets

## Strategy Description
This notebook implements a quantitative trading strategy based on the paper "Herding and Liquidity in Order-Book Markets. I. A Robust Liquidity-Stress Crossover and its Reflexive Mechanism" by Jan Novotny. The strategy focuses on identifying liquidity stress crossovers in the market using herding behavior and price momentum.

**Paper Citation:**
Novotny, J. (2026). Herding and Liquidity in Order-Book Markets. I. A Robust Liquidity-Stress Crossover and its Reflexive Mechanism. *arXiv preprint arXiv:2607.08907*.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

### Configuration
In this cell, we define the universe of tickers, parameters, and provide a hypothesis comment block.

In [ ]:
# Configuration
UNIVERSE = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']
HERDING_FRACTION = 0.9
HERDING_STRENGTH = 1.0
MOMENTUM_WINDOW = 20

# Hypothesis
"""
We hypothesize that by identifying periods of high herding behavior and price momentum,
we can capture liquidity stress crossovers that lead to profitable trading opportunities.
""

## Phase 2 — Data Download & Feature Computation

### Data Download
In this cell, we download market data for the defined universe using yfinance and compute the necessary features.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download data
data = yf.download(UNIVERSE, start='2020-01-01', end='2023-01-01')['Adj Close']

# Compute momentum
momentum = data.pct_change(MOMENTUM_WINDOW)

# Compute herding signal (simplified example)
def herding_signal(prices):
    return np.where(prices.diff() > 0, 1, -1)
herding = herding_signal(data)

# Cross-sectional normalization
momentum_normalized = momentum.sub(momentum.mean(axis=1), axis=0).div(momentum.std(axis=1), axis=0)

## Phase 3 — Signal Generation & Portfolio Construction

### Signal Generation
In this cell, we generate trading signals based on the herding and momentum features.

In [ ]:
# Generate signals
signals = np.where((herding == 1) & (momentum_normalized > 0), 1, -1)
signals = signals.shift(1)

### Position Sizing & Portfolio Construction
In this cell, we size positions and construct the portfolio based on the generated signals.

In [ ]:
# Position sizing
positions = signals.mul(1 / signals.abs().sum(axis=1), axis=0)

## Phase 4 — Vectorized Backtest

### Backtest
In this cell, we perform a vectorized backtest of the strategy, ensuring no look-ahead bias by shifting signals forward by 1 period.

In [ ]:
# Calculate daily returns
daily_returns = data.pct_change()

# Calculate portfolio returns
portfolio_returns = (daily_returns * positions).sum(axis=1)

## Phase 5 — Performance Metrics

### Performance Evaluation
In this cell, we calculate performance metrics such as Sharpe, Sortino, Calmar ratios, max drawdown, and plot the equity curve.

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import norm

# Calculate performance metrics
cumulative_returns = (1 + portfolio_returns).cumprod()
sharpe_ratio = np.sqrt(252) * portfolio_returns.mean() / portfolio_returns.std()
sortino_ratio = np.sqrt(252) * portfolio_returns.mean() / portfolio_returns[portfolio_returns < 0].std()
max_drawdown = (cumulative_returns.cummax() - cumulative_returns).max()
calmar_ratio = sharpe_ratio / max_drawdown

# Plot equity curve
plt.plot(cumulative_returns)
plt.title('Equity Curve')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns')
plt.show()

print(f'Sharpe Ratio: {sharpe_ratio:.2f}')
print(f'Sortino Ratio: {sortino_ratio:.2f}')
print(f'Calmar Ratio: {calmar_ratio:.2f}')
print(f'Max Drawdown: {max_drawdown:.2f}')

## Phase 6 — Monitoring Stub

### Monitoring Function
In this cell, we define a function that prints daily P&L and current positions given live data.

In [ ]:
# Monitoring function
def monitor_positions(data, positions):
    daily_returns = data.pct_change().dropna()
    portfolio_returns = (daily_returns * positions).sum(axis=1)
    daily_pnl = portfolio_returns.iloc[-1]
    current_positions = positions.iloc[-1]
    print(f'Daily P&L: {daily_pnl:.2f}')
    print('Current Positions:')
    print(current_positions)

# Example usage
monitor_positions(data, positions)